In [1]:
import pandas as pd
import json
from pathlib import Path

In [9]:
# Load the processed dataset locally
df = pd.read_csv(
    "../data/processed/customer_reviews_product_clustered.csv"
)
df.head(1)

,product_name,asins,brand,categories,manufacturer,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.title,reviews.text,clean_text,sentiment,text_length,asin_clean,valid_asin,product_name_clean,product_id,cluster,meta_category
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13 00:00:00+00:00,yes,0.0,5,Kindle,This product so far has not disappointed. My c...,this product so far has not disappointed. my c...,positive,143,B01AHB9CN2,True,all new fire hd 8 tablet 8 hd display wi fi 16...,ASIN_B01AHB9CN2,3,Kindle & Fire Devices – Special Offers


In [4]:
# Store only aggregated summaries in the reports folder
output_dir = Path("../reports/summaries_data")
output_dir.mkdir(parents=True, exist_ok=True)

In [10]:
# 1. Overall sentiment distribution
sentiment_summary = (
    df["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

sentiment_summary["percentage"] = (
    sentiment_summary["count"] / len(df) * 100
).round(2)

sentiment_summary.to_json(
    output_dir / "sentiment_distribution.json",
    orient="records",
    indent=2
)

In [12]:
# 2. Average rating and review count by category
category_ratings = (
    df.groupby("meta_category")
    .agg(
        avg_rating=("reviews.rating", "mean"),
        review_count=("reviews.rating", "count")
    )
    .reset_index()
)

category_ratings["avg_rating"] = (
    category_ratings["avg_rating"].round(2)
)

category_ratings.to_json(
    output_dir / "category_ratings.json",
    orient="records",
    indent=2
)


In [14]:
# 3. Sentiment distribution within each category
sentiment_by_category = (
    df.groupby(["meta_category", "sentiment"])
    .size()
    .reset_index(name="count")
)

sentiment_by_category["percentage"] = (
    sentiment_by_category["count"]
    / sentiment_by_category.groupby("meta_category")["count"].transform("sum")
    * 100
).round(2)

sentiment_by_category.to_json(
    output_dir / "sentiment_by_category.json",
    orient="records",
    indent=2
)

print("JSON summaries saved to:", output_dir)
print("Files:", [p.name for p in output_dir.glob("*.json")])

JSON summaries saved to: ../reports/summaries_data
Files: ['fire_kids_edition_tablets_accessories.json', 'category_ratings.json', 'echo_kindle_fire_devices.json', 'amazon_device_accessories_chargers.json', 'kindle_fire_devices_special_offers.json', 'sentiment_distribution.json', 'category_summary.json', 'sentiment_by_category.json', 'echo_smart_home_devices.json', 'kindle_fire_tablets_16gb.json']
